# ClimateLens: Typhoon QLoRA
Notebook สำหรับทำซ้ำและตรวจผลการทดลองแยกจากเว็บ ใช้ NVIDIA GPU บน Google Colab

เลือก `MODE = "restore"` เพื่อเปิด artifacts ที่ดาวน์โหลดไว้โดยไม่ฝึกใหม่ หรือ `"train"` เพื่อทำซ้ำ 4 configurations บนข้อมูลที่ตรวจแล้ว
เซลล์ฝึกจะไม่ทำงานในโหมด restore ส่วน Final test และชุดท้าทายต้องเปิดใช้งานแยก

ผลที่เคยรายงาน: validation 16 ข้อได้ Accuracy/F1 = 1.0 ทั้ง base และ adapter ทุก run; ชุดท้าทาย 80 ข้อได้ Accuracy 0.9625 และ Macro F1 0.9624 ทั้งคู่ จึงเลือกใช้ base แบบ zero-shot
ผลนี้เป็นข้อสรุปจากการทดลองเดิม ไม่ใช่ผลที่ notebook ฉบับนี้รันใหม่
เฉลยสังเคราะห์จาก AI ต้องตรวจยืนยันก่อนอ้างเป็นข้อมูลที่มนุษย์ตรวจแล้ว


## 1. เตรียม Colab
เปิด Runtime > Change runtime type > GPU แล้วอัปโหลด `climatelens_finetuning_colab.zip` รุ่นที่มากับ notebook นี้
สร้าง bundle ใหม่หลังแก้โค้ดด้วย `python -m categorization.finetuning.package_colab --output outputs/climatelens_finetuning_colab_clean.zip`
อัปโหลดเฉพาะ code bundle และข้อมูลที่อนุญาตให้ใช้บน Colab


In [ ]:
import io
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

import torch
from google.colab import files

assert torch.cuda.is_available(), "เลือก NVIDIA GPU ก่อน"
print("GPU:", torch.cuda.get_device_name(0))

def upload_one(suffix):
    uploaded = files.upload()
    names = [name for name in uploaded if name.lower().endswith(suffix)]
    if len(names) != 1:
        raise ValueError(f"เลือกไฟล์ {suffix} เพียงไฟล์เดียว")
    return uploaded[names[0]]

def extract_zip(content, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        for item in archive.infolist():
            if not (destination / item.filename).resolve().is_relative_to(destination):
                raise ValueError("Invalid ZIP path")
        archive.extractall(destination)

PROJECT = Path("/content/climatelens_finetuning")
extract_zip(upload_one(".zip"), PROJECT)
os.chdir(PROJECT)

def run_module(module, *arguments):
    subprocess.run(
        [sys.executable, "-m", module, *map(str, arguments)], check=True,
    )

run_module("pip", "install", "-r", "categorization/finetuning/requirements-gpu.txt")
import pandas as pd


## 2. กำหนดการทดลองในจุดเดียว
Run 01–02 ใช้ LR 1e-4 และ Run 03–04 ใช้ 5e-5 ตามแผนที่สอดคล้องกับ logs เดิม
ฉบับนี้ส่ง LR ผ่านคำสั่งโดยตรงและบันทึกใน run.json สำหรับการฝึกครั้งใหม่ ไม่แก้ metadata ของผลเก่า


In [ ]:
MODE = "restore"  # "restore" = เปิด artifacts เดิม, "train" = ฝึกใหม่
RUNS = {
    "01": {"learning_rate": 1e-4, "epochs": 2},
    "02": {"learning_rate": 1e-4, "epochs": 1},
    "03": {"learning_rate": 5e-5, "epochs": 2},
    "04": {"learning_rate": 5e-5, "epochs": 1},
}
SELECTED_RUN = "04"  # เลือกจาก validation ก่อนดู final test
METHOD = "zero-shot"
RUN_SMOKE = False
RUN_FINAL_TEST = False
RUN_CHALLENGE = False

assert MODE in {"restore", "train"}
assert SELECTED_RUN in RUNS
RUN = Path(f"outputs/typhoon_run_{SELECTED_RUN}")
METRIC_COLUMNS = [
    "accuracy", "macro_f1", "invalid_rate", "mean_seconds", "p95_seconds",
]

def show_metrics(directory):
    path = Path(directory) / "metrics.json"
    if not path.exists():
        print("ยังไม่มีผล:", path)
        return
    report = json.loads(path.read_text())
    display(pd.DataFrame(report["metrics"]).T[METRIC_COLUMNS])


## 3. เปิดผลเดิม หรือเตรียมข้อมูลฝึกใหม่
- restore: อัปโหลด `climatelens_finetuning_artifacts.zip` ที่มี outputs/typhoon_run_*/adapter/ และ run.json
- train: อัปโหลด CSV ที่คนตรวจแล้ว มี id, feedback, category, group_id ครบ 8 หมวด; ถ้ามีคอลัมน์ review ต้องใช้ reviewed_category ที่ยืนยันแล้วและ review_status = approved
ข้อความดัดแปลงจากสถานการณ์เดียวกันต้องใช้ group_id เดียวกัน การเตรียมข้อมูลไม่เปลี่ยนสถานะตรวจให้โดยอัตโนมัติ


In [ ]:
Path("outputs").mkdir(exist_ok=True)
if MODE == "restore":
    extract_zip(upload_one(".zip"), PROJECT)
    assert (RUN / "run.json").exists(), f"ไม่พบ {RUN / 'run.json'}"
    assert (RUN / "adapter").is_dir(), "ไม่พบ adapter ใน artifacts"
else:
    labelled = Path("outputs/labelled.csv")
    labelled.write_bytes(upload_one(".csv"))
    run_module(
        "categorization.finetuning.prepare", "--input", labelled,
    )


## 4. ฝึก 4 configurations
แต่ละ run เริ่มจาก base เดียวกัน เลือก checkpoint ด้วย validation loss และไม่ใช้ test ระหว่างฝึก
ข้ามทั้งส่วนเมื่อ MODE เป็น restore; smoke เป็นตัวเลือกสำหรับตรวจระบบเท่านั้น
โฟลเดอร์ run ที่มีอยู่แล้วจะหยุดเพื่อไม่เขียนทับผลเดิม


In [ ]:
if MODE == "train":
    if RUN_SMOKE:
        run_module(
            "categorization.finetuning.train",
            "--output", "outputs/typhoon_smoke", "--max-steps", 20,
        )
    for number, config in RUNS.items():
        run_module(
            "categorization.finetuning.train",
            "--output", f"outputs/typhoon_run_{number}",
            "--learning-rate", config["learning_rate"],
            "--epochs", config["epochs"],
        )
else:
    print("โหมด restore: ไม่ฝึกใหม่")


## 5. Validation และตารางรวม
เปรียบเทียบ base/adapter แบบ zero-shot ทุก run และ few-shot เฉพาะ Run 01 ตามการทดลองเดิม
โหลดผลที่มีอยู่แล้วโดยไม่ประเมินซ้ำ; โหมด restore แสดงเฉพาะผลที่อยู่ใน artifacts
NF4 ใน Transformers ไม่บังคับ JSON Schema จึงรายงานแยกจาก Ollama Q4_K_M


In [ ]:
evaluations = [
    ("01", "zero-shot", "validation_zero-shot"),
    ("01", "few-shot", "validation_few-shot"),
    *[(number, "zero-shot", f"validation_run_{number}") for number in ["02", "03", "04"]],
]
summary = []
for number, method, folder in evaluations:
    output = Path("outputs") / folder
    if MODE == "train" and not output.exists():
        run_module(
            "categorization.finetuning.evaluate",
            "--run", f"outputs/typhoon_run_{number}",
            "--method", method, "--output", output,
        )
    path = output / "metrics.json"
    if path.exists():
        report = json.loads(path.read_text())
        for variant, metrics in report["metrics"].items():
            summary.append({
                "run": number, "method": method, "model": variant,
                **{key: metrics[key] for key in METRIC_COLUMNS},
            })
display(pd.DataFrame(summary))


## 6. Final test ของ run ที่เลือก
ค่าเริ่มต้นไม่เปิด test ใหม่ แต่แสดงผลเดิมถ้ามี
เปิด RUN_FINAL_TEST เฉพาะเมื่อหยุดปรับทุกอย่างและยังไม่ได้ใช้ชุด test เลือกโมเดล/ค่า
ชุด test เดิมที่เคยเปิดผลแล้วไม่ถือเป็น unseen test สำหรับการทดลองครั้งใหม่
การประเมินใหม่ต้องมี prepared splits เดิมที่ hash ตรงกัน; artifacts รุ่นเก่าที่มีเฉพาะ adapter/metrics ใช้ดูผลเดิมได้


In [ ]:
final_output = Path(f"outputs/final_test_run_{SELECTED_RUN}")
if RUN_FINAL_TEST:
    if final_output.exists():
        print("ใช้ผลเดิม:", final_output)
    else:
        run_module(
            "categorization.finetuning.evaluate",
            "--run", RUN, "--split", "test", "--confirm-final-test",
            "--method", METHOD, "--output", final_output,
        )
show_metrics(final_output)


## 7. ชุดท้าทายเพิ่มเติม: base เทียบ adapter ในครั้งเดียว
เปิด RUN_CHALLENGE แล้วอัปโหลด CSV 80 ข้อ (หรือชุดใหม่) ที่มี id, feedback และ reviewed_category
หากไม่มี reviewed_category จะใช้ category; ระบบบันทึกสถานะการตรวจตามไฟล์ ไม่อ้างว่า AI review คือ human approval
ชุดนี้ใช้พัฒนาเพิ่มเติม แยกจาก final test ไม่ต้อง prepare หรือฝึกใหม่
มีเพียง feedback ถูกส่งเข้าโมเดล ส่วนเฉลยใช้คำนวณคะแนนภายหลัง


In [ ]:
challenge_output = Path(f"outputs/challenge_run_{SELECTED_RUN}")
if RUN_CHALLENGE and not challenge_output.exists():
    challenge_csv = Path("outputs/challenge.csv")
    challenge_csv.write_bytes(upload_one(".csv"))
    run_module(
        "categorization.finetuning.evaluate",
        "--run", RUN, "--input-csv", challenge_csv,
        "--method", "zero-shot", "--output", challenge_output,
    )

if (challenge_output / "metrics.json").exists():
    show_metrics(challenge_output)
    predictions = pd.read_csv(
        challenge_output / "predictions.csv", encoding="utf-8-sig",
    )
    comparison = predictions.loc[
        predictions["variant"] == "base", ["id", "feedback", "answer", "predicted"],
    ].rename(columns={"predicted": "base_predicted"})
    adapter = predictions.loc[
        predictions["variant"] == "adapter", ["id", "predicted"],
    ].rename(columns={"predicted": "adapter_predicted"})
    comparison = comparison.merge(adapter, on="id", validate="one_to_one")
    base_correct = comparison["answer"].eq(comparison["base_predicted"])
    adapter_correct = comparison["answer"].eq(comparison["adapter_predicted"])
    subsets = {
        "fixed": comparison.loc[~base_correct & adapter_correct],
        "regressed": comparison.loc[base_correct & ~adapter_correct],
        "both_wrong": comparison.loc[~base_correct & ~adapter_correct],
    }
    comparison.to_csv(challenge_output / "comparison.csv", index=False, encoding="utf-8-sig")
    for name, frame in subsets.items():
        print(f"{name}: {len(frame)} ข้อ")
        display(frame)
        frame.to_csv(challenge_output / f"{name}.csv", index=False, encoding="utf-8-sig")
else:
    print("ข้ามชุดท้าทาย: เปิด RUN_CHALLENGE เมื่อต้องการทดลอง")


## 8. ดาวน์โหลดผลครั้งเดียว
รวม adapter, metadata, ผล validation/final/challenge และ manifest ของข้อมูลฝึกถ้ามี
ไม่รวม checkpoints, labelled.csv หรือ prepared splits
ผลชุดท้าทายมีข้อความ feedback ใน predictions/comparison จึงใช้เฉพาะข้อมูลที่อนุญาตให้ส่งออก
ดาวน์โหลด notebook เป็น .ipynb แยกต่างหาก; ฉบับสำหรับ Git ล้าง outputs ของเซลล์ก่อน commit


In [ ]:
zip_path = Path("climatelens_finetuning_artifacts.zip")
paths = set()
for run in Path("outputs").glob("typhoon_run_*"):
    paths.update((run / "adapter").rglob("*"))
    paths.update(run / name for name in [
        "run.json", "validation_loss.json", "log_history.json",
    ])
for directory in Path("outputs").iterdir():
    if directory.is_dir() and directory.name.startswith(
        ("validation", "final_test", "challenge_run_")
    ):
        paths.update(p for p in directory.rglob("*") if p.suffix in {".json", ".csv"})
manifest = Path("categorization/data/finetuning/manifest.json")
if manifest.exists():
    paths.add(manifest)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(paths):
        if path.is_file():
            archive.write(path, str(path))
print("ไฟล์ผลลัพธ์ที่รวม:", sum(path.is_file() for path in paths))
files.download(str(zip_path))
